# Predicting Clinical Trial Outcomes from Protocol Text

**A classical machine learning approach**

Machine Learning final project, May 2026.

---

## 1. Introduction

A clinical trial is a scientific study that tests whether a medical intervention
(a drug, a device, a behavioural program) is safe and effective in people. Before
a trial starts, its sponsor publishes a **protocol**: a structured document that
describes the condition under study, the intervention, who can take part
(the eligibility criteria), and what will be measured.

Many trials never reach a successful conclusion. Some are **terminated**,
**withdrawn**, or **suspended** because of low enrollment, safety concerns,
funding problems, or strategic decisions. A trial that stops early wastes money,
delays useful treatments, and exposes participants to risk without producing
usable evidence.

This project asks a simple but practical question:

> **Can we predict, from the information available about a trial, whether it will
> be *completed* or *terminated/withdrawn/suspended*?**

We treat this as a **binary text + tabular classification** problem and solve it
with classical machine learning models.

## 2. Problem formulation

### 2.1 Why this matters

- **Cost.** A single late-phase clinical trial can cost tens or hundreds of
  millions of dollars. Early warning that a trial is at risk lets sponsors
  reallocate resources.
- **Patients.** Participants accept risk by enrolling. A trial that stops without
  producing evidence breaks the implicit contract with them.
- **Evidence.** Terminated trials rarely publish results, creating gaps and bias
  in the medical literature.

Potential users of such a model include trial sponsors, contract research
organisations, regulators, and investors performing due diligence.

### 2.2 The learning task

Each example is a single trial. The label is derived from the trial's
`overall_status`:

$$
y =
\begin{cases}
1 & \text{if status} = \texttt{COMPLETED} \\
0 & \text{if status} \in \{\texttt{TERMINATED}, \texttt{WITHDRAWN}, \texttt{SUSPENDED}\}
\end{cases}
$$

Trials with non-final statuses (e.g. `RECRUITING`, `ACTIVE`) are excluded, because
their outcome is not yet known.

### 2.3 Assumptions and constraints

- The protocol text and metadata are written largely **before** the outcome is
  known. This is what makes prediction meaningful.
- **Data leakage risk.** In practice, records of terminated trials are often
  updated *after* the fact, and may contain explicit phrases such as
  *"study was terminated due to low enrollment"*. Using such text would let the
  model read the answer instead of predicting it. We address this explicitly in
  the preprocessing section.
- The dataset is in English only; findings may not transfer to other registries.

## 3. Data source and ethics

### 3.1 Source

Data comes from [**ClinicalTrials.gov**](https://clinicaltrials.gov/), the public
registry maintained by the U.S. National Library of Medicine. We use its
[API v2](https://clinicaltrials.gov/data-api/api), which returns study records as
structured JSON.

The download is handled by [`src/fetch_data.py`](../src/fetch_data.py), which keeps
only the fields we need and stores them as a local CSV:

```bash
python -m src.fetch_data --limit 2000 --output data/raw/trials.csv
```

For each trial we keep the text fields (`brief_summary`, `detailed_description`,
`eligibility_criteria`), tabular fields (`study_type`, `phase`,
`enrollment_count`, `conditions`, `intervention_types`), the label source
(`overall_status`), and `why_stopped` (kept **only** for leakage analysis, never
used as a feature).

### 3.2 Legal and ethical notes

- The registry is **public** and contains **no personally identifiable patient
  data (no PHI)**.
- Raw data is **not** committed to the repository. Only the download script is
  versioned, so anyone can reproduce the dataset. This follows good practice for
  data-heavy projects.

## 4. Approach overview

The rest of the notebook follows these steps:

1. **Preprocessing** - build the binary label, merge text fields, and remove
   leakage-prone content.
2. **Exploratory data analysis (EDA)** - class balance, text length, missingness.
3. **Mathematical background** - TF-IDF, logistic loss, regularization, metrics.
4. **Models** - logistic regression baseline, then linear SVM, random forest, and
   gradient boosting (XGBoost).
5. **Ablation and evaluation** - text-only vs tabular-only vs hybrid features,
   with stratified cross-validation and metrics suited to class imbalance.
6. **Results, error analysis, and limitations.**